In [19]:
from pathlib import Path
import os, json, random, numpy as np, pandas as pd
from tqdm.auto import tqdm

RAW_DATA       = "../clean_data/out_concat_clean.jsonl"        # ← cambia aquí
PAIRS_OUT_DIR  = "data/pairs_math"
RESULTS_DIR    = Path("results_nb")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

METHOD   = "sft"          # "sft" | "ppl-gap" | "dpo"
EPOCHS   = 2
BSZ      = 8
FP16     = True



In [20]:
MODELS = [
 "HuggingFaceTB/SmolLM2-135M",
 "HuggingFaceTB/SmolLM2-135M-Instruct",
 "Qwen/Qwen3-0.6B",
 "Qwen/Qwen2.5-0.5B-Instruct",
 "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
 "google/gemma-3-1b-it",
 "meta-llama/Llama-3.2-1B"
]

# %% -----------------------------------------------------------
# 1) Genera pares (solo la 1.ª vez) ----------------------------
from build_preference_dataset import build_preference_pairs
if not Path(PAIRS_OUT_DIR).exists():
    build_preference_pairs(RAW_DATA, PAIRS_OUT_DIR)


In [21]:
# %% -----------------------------------------------------------
# 2) Importa utilidades del pipeline ---------------------------
from datasets import load_from_disk, disable_caching
from transformers import AutoTokenizer, AutoModelForCausalLM
from trainer_factory import build_trainer
from evaluate import run_eval, extract_num          # de los módulos previos
disable_caching()

ds = load_from_disk(PAIRS_OUT_DIR)                 # DatasetDict
print(ds)



DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 12890
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1433
    })
})


In [22]:
# %% -----------------------------------------------------------
# Helper: tokeniza dataset para un modelo ----------------------
def tokenize_dataset(dataset, tokenizer, max_len=512):
    def tok_map(ex):
        ex["input_ids_chosen"] = tokenizer(
            f"{ex['prompt']}\n{ex['chosen']}",
            truncation=True, max_length=max_len).input_ids
        ex["input_ids_rejected"] = tokenizer(
            f"{ex['prompt']}\n{ex['rejected']}",
            truncation=True, max_length=max_len).input_ids
        return ex
    return dataset.map(tok_map, remove_columns=[])



In [ ]:
# %% -----------------------------------------------------------
# Bucle principal modelos → baseline → train → eval ------------
summaries = []

model_name = MODELS[0]
tag = model_name.split("/")[-1].replace(".", "_")
print(f"\n### Modelo: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
ds_tok = tokenize_dataset(ds, tokenizer)

sample_idx = 0
ejemplo = ds_tok["train"][sample_idx]

print("🔎  EJEMPLO DE ENTRENAMIENTO")
print("-" * 60)
print("Prompt:")
print(ejemplo["prompt"])
print("\nChosen (explicación):")
print(ejemplo["chosen"])
print("\nRejected (respuesta corta):")
print(ejemplo["rejected"])
print("-" * 60)


# ----- baseline ------------------------------------------------
baseline_path = RESULTS_DIR / f"{tag}_baseline.json"
if not baseline_path.exists():
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype="auto", device_map="auto",
        trust_remote_code=True)
    run_eval(base_model, tokenizer, ds_tok["test"], baseline_path)

# ----- fine-tune ----------------------------------------------
trainer = build_trainer(model_name, METHOD, ds_tok,
                        RESULTS_DIR / f"{tag}_{METHOD}",
                        tokenizer, epochs=EPOCHS,
                        bsz=BSZ, fp16=FP16)
trainer.train()
trainer.save_model(RESULTS_DIR / f"{tag}_{METHOD}")

# ----- evaluación post-FT -------------------------------------
ft_path = RESULTS_DIR / f"{tag}_{METHOD}_eval.json"
summary = run_eval(trainer.model, tokenizer, ds_tok["test"], ft_path)
summary.update({"model": tag, "method": METHOD})
summaries.append(summary)
print(summary)




### Modelo: HuggingFaceTB/SmolLM2-135M


Map: 100%|██████████| 1433/1433 [00:02<00:00, 593.42 examples/s]


🔎  EJEMPLO DE ENTRENAMIENTO
------------------------------------------------------------
Prompt:
Una cierta organización consiste en cinco líderes y un cierto número de miembros regulares. Cada año, los líderes actuales son expulsados de la organización. A continuación, cada miembro regular debe encontrar a dos nuevas personas para que se unan como miembros regulares. Finalmente, cinco nuevas personas son elegidas de fuera de la organización para convertirse en líderes. Al principio hay quince personas en la organización en total. ¿Cuántas personas en total habrá en la organización cinco años de ahora?

Chosen (explicación):
Sabemos que inicialmente hay 15 personas en la organización y 5 son líderes, así que hay $15 - 5 = 10$ miembros regulares.

El número de miembros regulares triplica cada año porque cada miembro regular añade 2 más miembros.

Vamos a encontrar el número de miembros regulares después de 5 años.

Inicialmente hay 10 miembros regulares.

Después de 1 año, hay $10 \time

Eval:  12%|█▏        | 173/1433 [07:24<52:54,  2.52s/it]Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
